# Credit Risk Scoring v2 - Mejora versión anterior
## Contexto 
Creamos una herramienta que nos sirve para detectar posibles defaults con respecto a los clientes solventes. Es importante, tener detectados a los clientes defaults para evitar posibles pérdidas de capital para una entidad financiera. Si rechazamos a un buen cliente, tenemos cero riesgo, pero también es crítico por posibles fugas de clientes con los que podríamos generar beneficio a medio/largo plazo, conceptualmente de coste de oportunidad.

## Dataset
El conjunto de datos se extrae de un dataset público del German Credit Data en OpenML.


# Imports

In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance

# Carga de datos

In [2]:
credit = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
X = credit.data
y = credit.target

### Visualización de las dimensiones del dataset

In [3]:
print("Dimensión de X:",X.shape)
print("Dimensión de y:",y.shape)

Dimensión de X: (1000, 20)
Dimensión de y: (1000,)


### Vista previa y balance del target

In [4]:
print(X.head())
print(y.value_counts(normalize=True))  #sin normalize=True nos da valores absolutos, pero con este parámetro te devuelve la proporción

  checking_status  duration                  credit_history  \
0              <0         6  critical/other existing credit   
1        0<=X<200        48                   existing paid   
2     no checking        12  critical/other existing credit   
3              <0        42                   existing paid   
4              <0        24              delayed previously   

               purpose  credit_amount    savings_status employment  \
0             radio/tv           1169  no known savings        >=7   
1             radio/tv           5951              <100     1<=X<4   
2            education           2096              <100     4<=X<7   
3  furniture/equipment           7882              <100     4<=X<7   
4              new car           4870              <100     1<=X<4   

   installment_commitment     personal_status other_parties  residence_since  \
0                       4         male single          none                4   
1                       2  female div/de

### Balance del target

* El resultado de los targets 70/30 significa que existe un desbalanceo moderado de clientes. 
* Si de 1000 clientes tenemos 300 morosos, según los importes puede afectar a la solvencia de la entidad. 
* El accuracy es engañoso, ya que tenga un porcentaje de acierto de un 70% no significa que no haya riesgo, porque no tener en cuenta el 30% restante puede implicar que exista un riesgo de default loss que se debe tener en cuenta. 


## Selección de variables

In [5]:
X.columns

Index(['checking_status', 'duration', 'credit_history', 'purpose',
       'credit_amount', 'savings_status', 'employment',
       'installment_commitment', 'personal_status', 'other_parties',
       'residence_since', 'property_magnitude', 'age', 'other_payment_plans',
       'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone',
       'foreign_worker'],
      dtype='object')

He escogido las siguientes variables:

1. duration - duración del préstamo.
2. credit_amount - importe solicitado.
3. credit_history - historial crediticio.
4. checking_status - estado cuenta corriente.
5. savings_status - nivel de ahorros.
6. employment - antigüedad laboral.

In [6]:
X_sel = X[['duration', 'credit_amount', 'credit_history', 'checking_status', 'savings_status', 'employment']]

In [7]:
print(X_sel.shape)
X_sel.head()

(1000, 6)


,duration,credit_amount,credit_history,checking_status,savings_status,employment
0,6,1169,critical/other existing credit,<0,no known savings,>=7
1,48,5951,existing paid,0<=X<200,<100,1<=X<4
2,12,2096,critical/other existing credit,no checking,<100,4<=X<7
3,42,7882,existing paid,<0,<100,4<=X<7
4,24,4870,delayed previously,<0,<100,1<=X<4


### One Hot Encoding

In [8]:
X_encoded = pd.get_dummies(X_sel, drop_first=True)

In [9]:
print(X_encoded.shape)
X_encoded.head()

(1000, 17)


,duration,credit_amount,credit_history_critical/other existing credit,credit_history_delayed previously,credit_history_existing paid,credit_history_no credits/all paid,checking_status_<0,checking_status_>=200,checking_status_no checking,savings_status_500<=X<1000,savings_status_<100,savings_status_>=1000,savings_status_no known savings,employment_4<=X<7,employment_<1,employment_>=7,employment_unemployed
0,6,1169,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False
1,48,5951,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False
2,12,2096,True,False,False,False,False,False,True,False,True,False,False,True,False,False,False
3,42,7882,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False
4,24,4870,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False


Hemos aplicado One Hot Encoding a las 6 variables seleccionadas (X_sel). Las 2 variables numéricas se quedan igual, pero las variables categóricas, se dividen en varias columnas borrando una opción y si se cumple se marca como True y si no, se marca como False.

## Train/Test/Split
parámetros que utilizamos: 
* test_size=0.2 -> reserva el 20% para test, 80% para train.
* random_state=42 -> para fijar la aleatoriedad. El 42 es una convención.
* stratify=y -> como los datos están desbalanceados, es obligatorio utilizarlo. 

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(800, 17)
(200, 17)
class
good    0.7
bad     0.3
Name: proportion, dtype: float64
class
good    0.7
bad     0.3
Name: proportion, dtype: float64


## Escalado de variables numéricas


### IMPORTANTE: ejecutar siempre después del split. No reejecutar sola (escalaría sobre datos ya escalados).

In [12]:
cols_numericas = ['duration', 'credit_amount']
scaler = StandardScaler()
X_train[cols_numericas] = scaler.fit_transform(X_train[cols_numericas])
X_test[cols_numericas] = scaler.transform(X_test[cols_numericas])
print(X_train[cols_numericas].head())

     duration  credit_amount
675  0.755149       0.485384
703  0.755149      -0.246578
12  -0.726746      -0.584573
845  0.014201       0.285331
795 -0.973728      -0.319522


* Se tuvo que escalar las variables duration y credit_amount, porque los valores tienen una media muy grande de valores. 

* Sólo se escalan las 2 numéricas, porque las booleanas con one-hot encoding no lo necesitan.

* Se ha aplicado fit_transform sobre fit porque aprende sobre la media y la desviación, en cambio transform se aplica sobre test porque aplica lo que ha aprendido fit_transform.

## Baseline: Logistic Regression

In [13]:
# Implementación Logistic Regression

modelo_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)
print(y_pred_lr)

['good' 'bad' 'bad' 'good' 'bad' 'good' 'good' 'bad' 'bad' 'good' 'good'
 'good' 'bad' 'bad' 'good' 'bad' 'bad' 'good' 'good' 'good' 'good' 'good'
 'good' 'good' 'good' 'good' 'good' 'bad' 'good' 'bad' 'good' 'good'
 'good' 'good' 'bad' 'bad' 'bad' 'bad' 'good' 'bad' 'good' 'good' 'good'
 'bad' 'bad' 'bad' 'bad' 'good' 'bad' 'good' 'good' 'bad' 'bad' 'good'
 'good' 'good' 'good' 'bad' 'good' 'good' 'good' 'good' 'good' 'good'
 'bad' 'good' 'good' 'bad' 'bad' 'good' 'good' 'bad' 'good' 'bad' 'good'
 'bad' 'good' 'bad' 'bad' 'bad' 'bad' 'bad' 'good' 'bad' 'bad' 'good'
 'bad' 'good' 'good' 'good' 'bad' 'good' 'bad' 'good' 'good' 'bad' 'bad'
 'bad' 'bad' 'bad' 'good' 'bad' 'good' 'bad' 'good' 'good' 'good' 'good'
 'bad' 'good' 'good' 'good' 'good' 'bad' 'good' 'good' 'good' 'good' 'bad'
 'good' 'bad' 'good' 'good' 'good' 'good' 'bad' 'good' 'good' 'good' 'bad'
 'good' 'bad' 'good' 'good' 'bad' 'good' 'bad' 'bad' 'bad' 'good' 'good'
 'good' 'good' 'bad' 'good' 'bad' 'bad' 'good' 'bad' 'bad'

## Evaluacion: Logistic Regression

In [14]:
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

[[45 15]
 [47 93]]
              precision    recall  f1-score   support

         bad       0.49      0.75      0.59        60
        good       0.86      0.66      0.75       140

    accuracy                           0.69       200
   macro avg       0.68      0.71      0.67       200
weighted avg       0.75      0.69      0.70       200



Como sklearn ordena alfabéticamente arriba tienes bad y abajo good. La matriz de confusión la interpretamos: 
* 45 fueron etiquetados como morosos correctamente.
* 15 fueron etiquetados como buen cliente, pero al final resultaron ser morosos. Punto principal a reforzar (FN), porque se deberían detectar correctamente o disminuir la probabilidad de que suceda.
* 47 fueron etiquetados como morosos y se le negó el préstamo, pero eran buenos clientes (coste de oportunidad) - este punto se debe reforzar porque es negativo para el prestamista, pero como segundo punto.
* 93 préstamos aprobados correctamente.

* recall 0.75 nos está indicando que de 60 morosos (45 fueron detectados y 15 no).
* precision 0.49 marcó 92 como morosos (45 reales y 47 falsos). 

* El modelo ha marcado como morosos a buenos clientes porque con class_weight='balanced' da más peso a los errores sobre la clase morosos que es la minoritaria durante el entrenamiento. Eso desplaza la frontera de decisión ya que el modelo se vuelve más propenso a marcar "bad" porque equivocarse con un moroso le duele más (Consecuencia: marca más morosos (recall alto) pero también marca buenos por error (precisión baja)). Para un banco, por ejemplo, que los buenos clientes son también importantes para no tener fugas hacia otras entidades por estos rechazos que al final se traduce en menos fondos y menos con lo que trabajar para generar como empresa y los morosos no detectados podrían ser una gran pérdida de capital.
 
* Si queremos ser un banco conservador, en época de crisis financiera, se rechaza mucho, pero si es una época de captación de cuota de mercado, prefieren asumir el riesgo de aceptar a posibles morosos por no perder clientes.

## Random Forest

In [15]:
# Creacion modelo random forest
modelo_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# Entrenamiento modelo
modelo_rf.fit(X_train, y_train)

# Predicción
y_pred_rf = modelo_rf.predict(X_test)

## Evaluacion: Random Forest

In [16]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[ 31  29]
 [ 21 119]]
              precision    recall  f1-score   support

         bad       0.60      0.52      0.55        60
        good       0.80      0.85      0.83       140

    accuracy                           0.75       200
   macro avg       0.70      0.68      0.69       200
weighted avg       0.74      0.75      0.74       200



El modelo Random Forest tiene mejor accuracy que el modelo Logistic Regression, pero para el análisis de morosidad si nos fijamos en la detección de morosos, RF es peor ya que ha dado préstamos a 29 morosos con respecto a los 15 que nos daría LR. Esto se debe a que Random Forest por mucho que le indiques balanced es más conservador a la hora de etiquetar a un moroso y eso implica que su recall sobre la clase 'bad' disminuye. Esto implica que hay un mayor riesgo de que conceda préstamos a un moroso por falta de detección. 

## Threshold tuning: validation set

In [17]:
# Creacion de un set de validacion sobre el train. 800 filas, le damos el 25% a validation (200) y el resto se queda en train.

X_train_final, X_val, y_train_final, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

In [18]:
print(X_train_final.shape)
print(X_val.shape)

(600, 17)
(200, 17)


# Reentrenamiento de Random Forest finetuneando el umbral

In [19]:
# Reentreno sobre las 600 filas y calculo de probabilidades
modelo_rf_v2 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
modelo_rf_v2.fit(X_train_final, y_train_final)

# Probabilidad sobre validation
probas_val = modelo_rf_v2.predict_proba(X_val)

print(probas_val[:5])


[[0.65 0.35]
 [0.7  0.3 ]
 [0.28 0.72]
 [0.46 0.54]
 [0.01 0.99]]


In [20]:
# Extraccion columna bad
probas_bad = probas_val[:,0]

print(probas_bad[:5])

[0.65 0.7  0.28 0.46 0.01]


La columna 0 corresponde a "bad". Esto se debe a sklearn ordena las clases alfabéticamente, y "bad" va antes que "good".

## Threshold tuning: probar con umbral 0.35

In [21]:
umbral = 0.35
y_pred_umbral = np.where(probas_bad >= umbral, 'bad', 'good')
print(confusion_matrix(y_val, y_pred_umbral))
print(classification_report(y_val, y_pred_umbral))

[[ 35  25]
 [ 32 108]]
              precision    recall  f1-score   support

         bad       0.52      0.58      0.55        60
        good       0.81      0.77      0.79       140

    accuracy                           0.71       200
   macro avg       0.67      0.68      0.67       200
weighted avg       0.73      0.71      0.72       200



Bajar el umbral hemos conseguido un incremento en el recall de bad y con ello conseguimos cazar más morosos.

## Threshold tuning: probar con lista de umbrales

In [22]:
# Barrido de umbrales

umbrales = [0.2, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

for umbral in umbrales:
    y_pred_umbral = np.where(probas_bad >= umbral, 'bad', 'good')
    recall_bad = recall_score(y_val, y_pred_umbral, pos_label='bad')
    precision_bad = precision_score(y_val, y_pred_umbral, pos_label='bad')
    print(f"Umbral {umbral}: recall={recall_bad: .2f}, precision={precision_bad: .2f}")



Umbral 0.2: recall= 0.82, precision= 0.44
Umbral 0.25: recall= 0.72, precision= 0.47
Umbral 0.3: recall= 0.65, precision= 0.49
Umbral 0.35: recall= 0.58, precision= 0.52
Umbral 0.4: recall= 0.48, precision= 0.54
Umbral 0.45: recall= 0.45, precision= 0.55
Umbral 0.5: recall= 0.32, precision= 0.53


Mi intuición inicial mirando solo recall/precision, antes de aplicar la matriz de costes. Tras ver esta tabla, me decantaría por el umbral 0.25 priorizando la caza de morosos con un recall 0.72 y con eso baja la precision que implica que molesto a buenos clientes, pero acepto ese coste, porque, para un banco, un moroso que no se detecta tiene un mayor coste que un cliente bueno rechazado.

## Matriz de costes

In [23]:
coste_fn = 7 # coste de un moroso colado
coste_fp = 3 # coste de un buen cliente rechazado

for umbral in umbrales:
    y_pred_umbral = np.where(probas_bad >= umbral, 'bad', 'good')
    matriz = confusion_matrix(y_val, y_pred_umbral)
    fn = matriz [0][1]  # morosos colados
    fp = matriz [1][0]  # buenos rechazados
    coste_total = fn * coste_fn + fp * coste_fp
    print(f"Umbral {umbral}: FN={fn}, FP={fp}, coste={coste_total}")
    print(matriz)

Umbral 0.2: FN=11, FP=62, coste=263
[[49 11]
 [62 78]]
Umbral 0.25: FN=17, FP=49, coste=266
[[43 17]
 [49 91]]
Umbral 0.3: FN=21, FP=40, coste=267
[[ 39  21]
 [ 40 100]]
Umbral 0.35: FN=25, FP=32, coste=271
[[ 35  25]
 [ 32 108]]
Umbral 0.4: FN=31, FP=25, coste=292
[[ 29  31]
 [ 25 115]]
Umbral 0.45: FN=33, FP=22, coste=297
[[ 27  33]
 [ 22 118]]
Umbral 0.5: FN=41, FP=17, coste=338
[[ 19  41]
 [ 17 123]]


Tras darle un peso de penalización indicado como coste para los morosos que no se detectan y a los clientes rechazados, podemos hacer una estimación de la pérdida que puede suponer para la entidad. Le puse un 7 como coste de los morosos no detectados (FN) y 3 a los clientes rechazados (FP) por error porque tomamos importancia a ambos grupos, pero no afectan lo mismo.

* Los valores con menor coste y mejores resultados estarían entre los umbrales 0.2 - 0.35

* Al final, tras ejecutarlo, me quedaría con el valor 0.3 porque hay una cantidad de morosos que se cuelan, pero se reduce bastante el no conceder préstamos a buenos clientes, por lo que mejora la reputación de la entidad, al menor coste posible.

# Validacion en TEST

In [24]:
# Calculamos las probabilidades sobre X_test
probas_test = modelo_rf_v2.predict_proba(X_test)

In [25]:
# Extraccion columna "bad"
probas_test_bad = probas_test[:,0]
print(probas_test_bad[:5])

[0.24 0.34 0.29 0.42 0.32]


In [26]:
# Predicciones con el umbral decidido (0.3)
umbral = 0.3

y_pred_test = np.where(probas_test_bad >= umbral,'bad', 'good')
print(confusion_matrix(y_test,y_pred_test))
print(classification_report(y_test,y_pred_test))

[[ 47  13]
 [ 40 100]]
              precision    recall  f1-score   support

         bad       0.54      0.78      0.64        60
        good       0.88      0.71      0.79       140

    accuracy                           0.73       200
   macro avg       0.71      0.75      0.71       200
weighted avg       0.78      0.73      0.75       200



Al ejecutarlo, nos está mejorando los resultados obtenidos con anterioridad. 
* Recall bad (morosos detectados) hemos pasado de un 0.65 a 0.78 por lo que hemos disminuido parte de la filtración. 
* Precision bad (de estos morosos cuantos lo eran realmente) hemos pasado del 0.49 a 0.54 por lo que hemos mejorado la detección real de morosos. 

* Podemos concluir que el modelo generaliza bien y no estaba sobreajustado sobre validation, pero ha sido un poco azar y no es un dato concluyente de que el modelo sea bueno.

# Metricas avanzadas
## ROC-AUC

El AUC mide la capacidad del modelo de separar clases a través de todos los umbrales.

Posibles valores de AUC y como interpretarlo:

* 0.5 o inferior = inútil
* 0.7 = aceptable
* 0.8 = bueno
* 0.9 = muy bueno
* 1.0 = sospechoso (suele ser leakage)

In [27]:
# Convertir y_test a numérico (bad=1, good=0)
y_test_num = (y_test == 'bad').astype(int)

# Calcular ROC-AUC usando las probabilidades
auc = roc_auc_score(y_test_num, probas_test_bad)
print(f"ROC-AUC: {auc: .3f}")

ROC-AUC:  0.814


Ha dado un resultado de AUC bueno aun teniendo 6 variables. Este valor indica que tenemos un modelo que generaliza bien y tiene buena capacidad de discriminacion. 

## KS (kolmogorov-Smirnov)
KS lo usamos para medir la maxima separacion entre dos distribuciones de score. El KS debería ser más bajo que el AUC, debería ser superior a 0.3 para considerarse un buen modelo para banca.

+ KS<0.2 = modelo débil
+ KS 0.2-0.3 = aceptable
+ KS 0.3-0.5 = bueno
+ KS > 0.5 = excelente

In [28]:
# Separacion probabilidad por clase real

probas_morosos = probas_test_bad[y_test == 'bad']
probas_buenos = probas_test_bad[y_test == 'good']

In [29]:
# Calcular KS
ks = ks_2samp(probas_morosos, probas_buenos)
print(f"KS statistic: {ks.statistic: .3f}")

KS statistic:  0.519


Este resultado ≈0.52 nos está indicando que el modelo separa muy bien los morosos de los buenos pagadores.

# Feature importance
Sirve a la hora de la interpretabilidad/compliance, ya que nos permite dar una explicación al resultado de denegado del crédito. También nos da que variables está utilizando para la validación y podremos saber si tiene sentido o no a la hora de validar el modelo.

Este método asigna un valor a cada columna, que en conjunto suman 1 y las que tienen mayor número son la que tuvieron más peso en la decisión. Eso sí, hay que ir con ojo porque no te da el nombre de la columna, sino que solo el número. 

Se puede utilizar una extracción del nombre de la columna primero y luego el valor en el resultado.

In [30]:
# proceso base de feature importance
importancias = pd.DataFrame({
    'variable': X_train.columns,
    'importancia': modelo_rf_v2.feature_importances_
})

importancias = importancias.sort_values('importancia', ascending=False)
print(importancias)

                                         variable  importancia
1                                   credit_amount     0.343537
0                                        duration     0.218037
8                     checking_status_no checking     0.080412
2   credit_history_critical/other existing credit     0.039589
6                              checking_status_<0     0.038537
15                                 employment_>=7     0.036390
10                            savings_status_<100     0.032844
14                                  employment_<1     0.030632
13                              employment_4<=X<7     0.029371
12                savings_status_no known savings     0.028606
4                    credit_history_existing paid     0.026232
3               credit_history_delayed previously     0.018414
16                          employment_unemployed     0.016486
9                      savings_status_500<=X<1000     0.016179
7                           checking_status_>=200     0

Con estos resultados podemos observar que para la prediccion y clasificación de clientes según si son morosos o no, las variables que ha tenido en cuenta son **credit_amount** y **duration** y en menor peso **checking_status_no checking**.

Las variables numéricas (importe, duración) dominan el ranking, pero puede estar afectado por el sesgo del MDI hacia variables con muchos valores únicos. 

El histórico crediticio está repartido en 3-4 columnas y si estuviesen sumadas sería un valor comparable con importe y duración. 

Una alternativa más robusta sería calcular permutation importance para no tener este sesgo.

## Permutation importance

In [31]:
resultado_perm = permutation_importance(
    modelo_rf_v2, X_test, y_test, n_repeats=10, random_state=42
)

importancias_perm = pd.DataFrame({
    'variable': X_test.columns,
    'importancia_perm': resultado_perm.importances_mean
})
importancias_perm = importancias_perm.sort_values('importancia_perm', ascending=False)
print(importancias_perm)

                                         variable  importancia_perm
0                                        duration            0.0775
8                     checking_status_no checking            0.0460
1                                   credit_amount            0.0310
12                savings_status_no known savings            0.0300
6                              checking_status_<0            0.0245
4                    credit_history_existing paid            0.0180
16                          employment_unemployed            0.0115
7                           checking_status_>=200            0.0110
2   credit_history_critical/other existing credit            0.0100
15                                 employment_>=7            0.0100
10                            savings_status_<100            0.0070
14                                  employment_<1            0.0015
9                      savings_status_500<=X<1000            0.0010
11                          savings_status_>=100

El MDI sobreestimaba específicamente **credit_amount** (de 34% a 3% de importancia relativa al recalcular con permutation importance). En cambio, **duration** resultó tener una importancia real incluso mayor de la que sugería MDI. **Checking_status_no checking** se confirma como predictor sólido en ambos métodos. Algunas categorías de **credit_history** muestran importancia marginal o negativa, sugiriendo que aportan poco al modelo actual.

## Exportar modelo para la app

In [32]:
import joblib

joblib.dump(modelo_rf_v2, 'modelo_riesgo_rf_v2.pkl')
joblib.dump(scaler, 'scaler_v2.pkl')
joblib.dump(list(X_train.columns), 'columnas_modelo_v2.pkl')
joblib.dump(0.3, 'umbral_v2.pkl')


['umbral_v2.pkl']

## Comprobaciones para el archivo app.py

In [33]:
print(X['checking_status'].unique())
print(X['credit_history'].unique())
print(X['savings_status'].unique())
print(X['employment'].unique())

['<0', '0<=X<200', 'no checking', '>=200']
Categories (4, object): ['0<=X<200', '<0', '>=200', 'no checking']
['critical/other existing credit', 'existing paid', 'delayed previously', 'no credits/all paid', 'all paid']
Categories (5, object): ['all paid', 'critical/other existing credit', 'delayed previously', 'existing paid', 'no credits/all paid']
['no known savings', '<100', '500<=X<1000', '>=1000', '100<=X<500']
Categories (5, object): ['100<=X<500', '500<=X<1000', '<100', '>=1000', 'no known savings']
['>=7', '1<=X<4', '4<=X<7', 'unemployed', '<1']
Categories (5, object): ['1<=X<4', '4<=X<7', '<1', '>=7', 'unemployed']


In [34]:
print(X['credit_amount'].describe())
print(X['duration'].describe())

count     1000.000000
mean      3271.258000
std       2822.736876
min        250.000000
25%       1365.500000
50%       2319.500000
75%       3972.250000
max      18424.000000
Name: credit_amount, dtype: float64
count    1000.000000
mean       20.903000
std        12.058814
min         4.000000
25%        12.000000
50%        18.000000
75%        24.000000
max        72.000000
Name: duration, dtype: float64


In [35]:
print("Clientes con credit_amount > 10000:", (X['credit_amount'] > 10000).sum())
print("Clientes con credit_amount > 12000:", (X['credit_amount'] > 12000).sum())
print("Clientes con credit_amount > 15000:", (X['credit_amount'] > 15000).sum())
print("Total clientes:", len(X))

Clientes con credit_amount > 10000: 40
Clientes con credit_amount > 12000: 21
Clientes con credit_amount > 15000: 5
Total clientes: 1000


In [36]:
print("Clientes con duration > 50:", (X['duration'] > 50).sum())
print("Clientes con duration > 60:", (X['duration'] > 60).sum())
print("Clientes con duration > 70:", (X['duration'] > 70).sum())
print("Total clientes:", len(X))

Clientes con duration > 50: 16
Clientes con duration > 60: 1
Clientes con duration > 70: 1
Total clientes: 1000


In [37]:
filtro = (
    (X['checking_status'] == 'no checking') &
    (X['savings_status'] == 'no known savings') &
    (X['credit_history'] == 'critical/other existing credit') &
    (X['employment'] == 'unemployed')
)
print("Clientes con esa combinación exacta:", filtro.sum())
print(X[filtro][['duration', 'credit_amount']].describe())

Clientes con esa combinación exacta: 1
       duration  credit_amount
count       1.0            1.0
mean       18.0         3229.0
std         NaN            NaN
min        18.0         3229.0
25%        18.0         3229.0
50%        18.0         3229.0
75%        18.0         3229.0
max        18.0         3229.0


## Limitaciones del modelo

El modelo tiene cobertura limitada sobre combinaciones específicas de variables, especialmente perfiles con múltiples señales de riesgo simultáneas. Con 1000 observaciones, muchas combinaciones concretas de las 6 variables están representadas por 1 o 0 casos reales, lo que hace que las predicciones en esas zonas sean poco fiables. Se limitaron los rangos de duration(≤48) y credit_amount(≤10000€) en la app para evitar los casos individualmente más extremos, aunque esto no garantiza representación suficiente en todas las combinaciones.